# LeetCode 1432 — Max Difference You Can Get From Changing an Integer bằng Python

Link đề: https://leetcode.com/problems/max-difference-you-can-get-from-changing-an-integer/ · Độ khó: **Medium** · Chủ đề: Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maxDiff(self, num: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Tách bài thành hai bài con **độc lập**: tìm số **lớn nhất** `a` và số **nhỏ nhất** `b` có thể tạo ra; đáp án `a − b`.
2. Viết **brute force** thử cả `10 × 10 = 100` cặp `(x, y)` — đủ nhanh vì `num ≤ 10^8` chỉ có ≤ 9 chữ số — và lọc kết quả không hợp lệ (số 0 đứng đầu, bằng 0).
3. Suy ra **tham lam** theo nguyên tắc "chữ số **bên trái** có trọng số lớn nhất": max ⇒ đổi chữ số đầu tiên khác 9 thành 9; min ⇒ đổi chữ số đầu thành 1, hoặc nếu đã là 1 thì đổi chữ số đầu tiên khác `0` và `1` thành 0.
4. Thao tác chữ số bằng **chuỗi** (`str.replace`) và bằng **số học** (`divmod`, `% 10`).
5. Tránh các lỗi: tạo số 0 đứng đầu, đổi chỉ lần xuất hiện đầu tiên, chọn sai chữ số để đổi.

## 2. Đề bài & ví dụ

Cho số nguyên `num`. Bạn thực hiện **hai lần riêng biệt** (mỗi lần bắt đầu lại từ `num` gốc) thao tác sau:
1. Chọn một chữ số `x` (`0 ≤ x ≤ 9`).
2. Chọn một chữ số `y` (`0 ≤ y ≤ 9`, có thể `y = x`).
3. Thay **mọi** lần xuất hiện của `x` trong biểu diễn thập phân của `num` bằng `y`.

Gọi `a`, `b` là kết quả của hai lần. Trả về **hiệu lớn nhất** `a − b`. Lưu ý: `a` và `b` **không được có chữ số 0 đứng đầu** và **không được bằng 0**.

**Ví dụ 1:** `num = 555` → `888` (`a = 999` với `x=5,y=9`; `b = 111` với `x=5,y=1`).

**Ví dụ 2:** `num = 9` → `8` (`a = 9`, `b = 1`).

**Ràng buộc:** `1 <= num <= 10^8`.

In [1]:
import random
import time
from typing import List

examples = [(555, 888), (9, 8)]
for num, want in examples:
    print(num, "->", want)

555 -> 888
9 -> 8


## 3. Kiến thức nền

### 3.1 Chuyển đổi số ↔ chuỗi và `str.replace`

`str(num)` cho chuỗi chữ số; `s.replace(x, y)` thay **mọi** lần xuất hiện (tham số thứ ba `count` giới hạn số lần thay — **đừng** dùng ở bài này). `int(s)` đổi lại thành số.

In [2]:
s = str(1101057)
print(s, s.replace("1", "9"), s.replace("1", "9", 1))
print(int("0057"), "<- int bỏ số 0 đứng đầu, nhưng đề cấm tạo ra chuỗi như vậy")

1101057 9909057 9101057
57 <- int bỏ số 0 đứng đầu, nhưng đề cấm tạo ra chuỗi như vậy


### 3.2 Duyệt chữ số và tìm phần tử đầu tiên thoả điều kiện

`next((c for c in s if cond(c)), default)` trả về phần tử đầu tiên thoả, hoặc `default` nếu không có — rất gọn để tìm "chữ số đầu tiên khác 9".

In [3]:
s = "9928"
first_not9 = next((c for c in s if c != "9"), None)
print(first_not9, next((c for c in "999" if c != "9"), None))

2 None


### 3.3 Tách chữ số bằng số học

`num % 10` là chữ số hàng đơn vị, `num // 10` bỏ chữ số đó. `divmod(num, 10)` cho cả hai. Ghép lại: `val = val * 10 + d`.

In [4]:
num, digits = 90210, []
while num:
    num, d = divmod(num, 10)
    digits.append(d)
digits.reverse()
print(digits)
val = 0
for d in digits:
    val = val * 10 + d
print(val)

[9, 0, 2, 1, 0]
90210


### 3.4 Giá trị vị trí (place value)

Trong số `d_1 d_2 … d_k`, chữ số `d_1` có trọng số `10^(k−1)` — **lớn hơn tổng** mọi chữ số phía sau cộng lại (`10^(k−1) > 9·(10^(k−2) + … + 1)`). Vì vậy muốn số lớn/nhỏ nhất, ưu tiên quyết định **chữ số bên trái nhất có thể thay đổi**.

In [5]:
k = 5
print(10 ** (k - 1), ">", 9 * sum(10 ** i for i in range(k - 1)))

10000 > 9999


## 4. Ý tưởng nền tảng / trực giác

Vì `a` và `b` được chọn **độc lập**, `max(a − b) = max(a) − min(b)`.

Thử tay `num = 1101057`:

**Tìm `a` lớn nhất.** Chữ số đầu là `1` ≠ 9. Đổi mọi `1` thành `9` ⇒ `9909057`. Có cách nào lớn hơn? Mọi cách khác giữ chữ số đầu ≤ 1… trừ khi đổi `x = 1`. Vậy `a = 9909057`.

**Tìm `b` nhỏ nhất.** Chữ số đầu đã là `1` — không thể nhỏ hơn (0 bị cấm). Giữ nguyên chữ số đầu ⇒ **không được** đổi `x = 1` (vì sẽ đổi luôn chữ số đầu). Tìm chữ số đầu tiên (từ trái) có thể giảm: bỏ qua `1` (không được đổi) và `0` (đã nhỏ nhất) ⇒ gặp `5` ⇒ đổi `5 → 0` ⇒ `1101007`.

⇒ Đáp án `9909057 − 1101007 = 8808050`.

Với `num = 555`: `a = 999`; chữ số đầu `5 ≠ 1` ⇒ đổi `5 → 1` ⇒ `b = 111` ⇒ `888` ✔.

In [6]:
num = 1101057
s = str(num)
a = int(s.replace("1", "9"))
b = int(s.replace("5", "0"))
print(a, b, a - b)

9909057 1101007 8808050


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp `(x, y)`

**Ý tưởng:** với mỗi `x, y ∈ 0..9`, tạo `t = s.replace(x, y)`. Hợp lệ nếu `t[0] != '0'` (không có số 0 đứng đầu — điều này cũng loại luôn trường hợp kết quả bằng 0). Thu tất cả giá trị hợp lệ, đáp án là `max − min`.

Chỉ có 100 cặp, mỗi cặp `O(d)` với `d ≤ 9` chữ số ⇒ rất nhanh. Đây là gợi ý chính thức của LeetCode.

In [7]:
def all_results(num):
    s = str(num)
    res = set()
    for x in "0123456789":
        for y in "0123456789":
            t = s.replace(x, y)
            if t[0] != "0":              # không có 0 đứng đầu, và t != "0"
                res.add(int(t))
    return res

def max_diff_brute(num):
    r = all_results(num)
    return max(r) - min(r)

for num, want in examples:
    print(num, max_diff_brute(num), "✔" if max_diff_brute(num) == want else "✘")
print("số kết quả hợp lệ của 1101057:", len(all_results(1101057)), "| max/min:", max(all_results(1101057)), min(all_results(1101057)))

555 888 ✔
9 8 ✔
số kết quả hợp lệ của 1101057: 36 | max/min: 9909057 1101007


**Độ phức tạp:** `100 · O(d)` = `O(d)` với hằng số 100; `d = ⌊log10 num⌋ + 1 ≤ 9`. Bộ nhớ `O(d)`.

**Nút thắt?** Với ràng buộc này brute force **đã đủ nộp**. Nhưng:
1. Nếu `num` có hàng nghìn chữ số (biến thể dạng chuỗi), 100 lần thay chuỗi vẫn ổn, nhưng phải so sánh số lớn — tốn hơn.
2. Phỏng vấn thường muốn thấy **lập luận tham lam** để chỉ cần **2 lần** thay.

In [8]:
t = time.perf_counter()
acc = 0
for num in range(1, 20001):
    acc += max_diff_brute(num)
print(f"brute cho 20000 số: {time.perf_counter() - t:.3f}s (checksum {acc})")

brute cho 20000 số: 0.308s (checksum 928821327)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tách đôi:** `a` và `b` độc lập ⇒ tối đa hoá `a`, tối thiểu hoá `b` riêng rẽ.

**Insight 2 — trọng số chữ số bên trái lớn nhất:** so sánh hai số cùng số chữ số (thao tác không đổi độ dài) ⇒ so theo thứ tự từ điển; chữ số **đầu tiên khác nhau** quyết định.

**Quy tắc cho `a` (max):**
- **Nếu** mọi chữ số đều là 9 **thì** `a = num`.
- **Ngược lại** gọi `x` = chữ số **đầu tiên khác 9**; đổi mọi `x` thành `9`. (Các chữ số trước `x` đều là 9 — giữ nguyên; tại vị trí `x` tăng lên 9 là tối đa; chọn `x` khác thì vị trí này không tăng được.)

**Quy tắc cho `b` (min):**
- **Nếu** chữ số đầu `s[0] ≠ '1'` **thì** đổi mọi `s[0]` thành `1` (không được thành 0).
- **Ngược lại** (`s[0] == '1'`): tìm chữ số **đầu tiên** `x` với `x ∉ {'0', '1'}`; đổi mọi `x` thành `0`. Không có ⇒ `b = num`.
  - Bỏ qua `'0'`: đã nhỏ nhất.
  - Bỏ qua `'1'`: đổi `1` sẽ đổi luôn chữ số đầu (thành 0 ⇒ cấm, thành số khác ⇒ to hơn).

**Bất biến:** thao tác không làm thay đổi số chữ số (vì cấm số 0 đứng đầu), nên so sánh số = so sánh chuỗi cùng độ dài.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam bằng chuỗi (`str.replace`)

**Từng đoạn:**
1. `s = str(num)`.
2. `a`: tìm `x` đầu tiên khác `'9'`; nếu có thì `s.replace(x, '9')`.
3. `b`: nếu `s[0] != '1'` thì `s.replace(s[0], '1')`; ngược lại tìm `x` đầu tiên không thuộc `"01"` và `s.replace(x, '0')`.
4. Trả `int(a) − int(b)`.

In [9]:
def max_value(num):
    s = str(num)
    for c in s:
        if c != "9":
            return int(s.replace(c, "9"))
    return num

def min_value(num):
    s = str(num)
    if s[0] != "1":
        return int(s.replace(s[0], "1"))
    for c in s[1:]:
        if c not in "01":
            return int(s.replace(c, "0"))
    return num

def max_diff_greedy(num):
    return max_value(num) - min_value(num)

for num, want in examples + [(1101057, 8808050)]:
    print(num, max_value(num), min_value(num), max_diff_greedy(num), max_diff_greedy(num) == want)

555 999 111 888 True
9 9 1 8 True
1101057 9909057 1101007 8808050 True


In [10]:
class SolutionGreedy:
    def maxDiff(self, num: int) -> int:
        s = str(num)
        a = s
        for c in s:
            if c != "9":
                a = s.replace(c, "9")
                break
        b = s
        if s[0] != "1":
            b = s.replace(s[0], "1")
        else:
            for c in s[1:]:
                if c not in "01":
                    b = s.replace(c, "0")
                    break
        return int(a) - int(b)

print([SolutionGreedy().maxDiff(n) for n, _ in examples])

[888, 8]


**Độ phức tạp:** `O(d)` thời gian (vài lần quét chuỗi), `O(d)` bộ nhớ. Chỉ 2 lần thay thay vì 100.

### Cách 3 — Tham lam bằng số học (không dùng chuỗi)

**Từng đoạn:**
1. Tách chữ số vào list `ds` (trái → phải) bằng `divmod`.
2. Hàm `remap(ds, x, y)` ghép lại số với mọi `x` đổi thành `y`.
3. Chọn `x` cho max/min theo đúng quy tắc ở mục 6.

Hữu ích khi phỏng vấn yêu cầu "không chuyển sang chuỗi".

In [11]:
def digits_of(num):
    ds = []
    while num:
        num, d = divmod(num, 10)
        ds.append(d)
    return ds[::-1]

def remap(ds, x, y):
    val = 0
    for d in ds:
        val = val * 10 + (y if d == x else d)
    return val

def max_diff_math(num):
    ds = digits_of(num)
    x_max = next((d for d in ds if d != 9), None)
    a = remap(ds, x_max, 9) if x_max is not None else num
    if ds[0] != 1:
        b = remap(ds, ds[0], 1)
    else:
        x_min = next((d for d in ds[1:] if d > 1), None)
        b = remap(ds, x_min, 0) if x_min is not None else num
    return a - b

for num, want in examples + [(1101057, 8808050)]:
    print(num, max_diff_math(num), max_diff_math(num) == want)

555 888 True
9 8 True
1101057 8808050 True


In [12]:
class SolutionMath:
    def maxDiff(self, num: int) -> int:
        ds = []
        n = num
        while n:
            n, d = divmod(n, 10)
            ds.append(d)
        ds.reverse()

        def remap(x: int, y: int) -> int:
            val = 0
            for d in ds:
                val = val * 10 + (y if d == x else d)
            return val

        x_max = next((d for d in ds if d != 9), None)
        a = remap(x_max, 9) if x_max is not None else num
        if ds[0] != 1:
            b = remap(ds[0], 1)
        else:
            x_min = next((d for d in ds[1:] if d > 1), None)
            b = remap(x_min, 0) if x_min is not None else num
        return a - b

print([SolutionMath().maxDiff(n) for n, _ in examples])

[888, 8]


**Độ phức tạp:** `O(d)` thời gian, `O(d)` bộ nhớ cho list chữ số.

### Cách 4 — Brute force có định hướng: chỉ thử `x` có mặt trong số

Thay vì 100 cặp, chỉ thử `x` là các chữ số **thực sự xuất hiện** và `y ∈ {0, 1, 9}` (vì tối ưu chỉ cần đổi thành 9 cho max, 1 hoặc 0 cho min). Đây là cầu nối giữa brute force và tham lam: số phép thử ≤ `|digits| · 3 ≤ 30`, vẫn tự động lọc trường hợp không hợp lệ — ít lý luận hơn, khó sai hơn.

In [13]:
def max_diff_targeted(num):
    s = str(num)
    vals = []
    for x in set(s):
        for y in "019":
            t = s.replace(x, y)
            if t[0] != "0":
                vals.append(int(t))
    return max(vals) - min(vals)

print([max_diff_targeted(n) for n, _ in examples], max_diff_targeted(1101057))

[888, 8] 8808050


**Vì sao `y ∈ {0,1,9}` đủ?** Kết quả lớn nhất phải có các chữ số đổi thành giá trị lớn nhất có thể (9). Kết quả nhỏ nhất đổi thành giá trị nhỏ nhất hợp lệ: 0 nếu không chạm chữ số đầu, 1 nếu chạm chữ số đầu. Mọi `y` khác đều bị một trong ba lựa chọn này "trội hơn".

## 8. TRACE từng bước

Trace cách 2 với `num = 1101057` và `num = 9288`:

| `num` | bước max | `a` | bước min | `b` | `a − b` |
|---|---|---|---|---|---|
| 1101057 | chữ số đầu khác 9: `1` ⇒ `1→9` | 9909057 | `s[0]='1'`; bỏ `1,0,1,0`; gặp `5` ⇒ `5→0` | 1101007 | 8808050 |
| 9288 | `9` bỏ qua; gặp `2` ⇒ `2→9` | 9988 | `s[0]='9' ≠ 1` ⇒ `9→1` | 1288 | 8700 |

In [14]:
def trace_greedy(num):
    s = str(num)
    print(f"num = {s}")
    a = s
    for i, c in enumerate(s):
        if c == "9":
            print(f"  [max] vị trí {i}: '{c}' đã là 9, bỏ qua")
            continue
        a = s.replace(c, "9")
        print(f"  [max] vị trí {i}: '{c}' khác 9 -> đổi mọi '{c}' thành '9' => a = {a}")
        break
    else:
        print("  [max] toàn số 9 -> a = num")
    if s[0] != "1":
        b = s.replace(s[0], "1")
        print(f"  [min] chữ số đầu '{s[0]}' != '1' -> đổi mọi '{s[0]}' thành '1' => b = {b}")
    else:
        b = s
        print("  [min] chữ số đầu đã là '1', không được đụng tới chữ số 1")
        for i, c in enumerate(s[1:], start=1):
            if c in "01":
                print(f"  [min] vị trí {i}: '{c}' thuộc {{0,1}}, bỏ qua")
                continue
            b = s.replace(c, "0")
            print(f"  [min] vị trí {i}: '{c}' -> đổi mọi '{c}' thành '0' => b = {b}")
            break
        else:
            print("  [min] không có chữ số nào giảm được -> b = num")
    print(f"  => {a} - {b} = {int(a) - int(b)}")
    return int(a) - int(b)

for num in [1101057, 9288, 555, 10000]:
    trace_greedy(num)

num = 1101057
  [max] vị trí 0: '1' khác 9 -> đổi mọi '1' thành '9' => a = 9909057
  [min] chữ số đầu đã là '1', không được đụng tới chữ số 1
  [min] vị trí 1: '1' thuộc {0,1}, bỏ qua
  [min] vị trí 2: '0' thuộc {0,1}, bỏ qua
  [min] vị trí 3: '1' thuộc {0,1}, bỏ qua
  [min] vị trí 4: '0' thuộc {0,1}, bỏ qua
  [min] vị trí 5: '5' -> đổi mọi '5' thành '0' => b = 1101007
  => 9909057 - 1101007 = 8808050
num = 9288
  [max] vị trí 0: '9' đã là 9, bỏ qua
  [max] vị trí 1: '2' khác 9 -> đổi mọi '2' thành '9' => a = 9988
  [min] chữ số đầu '9' != '1' -> đổi mọi '9' thành '1' => b = 1288
  => 9988 - 1288 = 8700
num = 555
  [max] vị trí 0: '5' khác 9 -> đổi mọi '5' thành '9' => a = 999
  [min] chữ số đầu '5' != '1' -> đổi mọi '5' thành '1' => b = 111
  => 999 - 111 = 888
num = 10000
  [max] vị trí 0: '1' khác 9 -> đổi mọi '1' thành '9' => a = 90000
  [min] chữ số đầu đã là '1', không được đụng tới chữ số 1
  [min] vị trí 1: '0' thuộc {0,1}, bỏ qua
  [min] vị trí 2: '0' thuộc {0,1}, bỏ qua
  [mi

## 9. Code nộp LeetCode

Dùng cách 2 (tham lam bằng chuỗi):

In [15]:
class Solution:
    def maxDiff(self, num: int) -> int:
        s = str(num)
        # a: đổi chữ số đầu tiên khác 9 thành 9
        a = s
        for c in s:
            if c != "9":
                a = s.replace(c, "9")
                break
        # b: chữ số đầu -> 1; nếu đã là 1 thì chữ số đầu tiên không thuộc {0,1} -> 0
        b = s
        if s[0] != "1":
            b = s.replace(s[0], "1")
        else:
            for c in s[1:]:
                if c not in "01":
                    b = s.replace(c, "0")
                    break
        return int(a) - int(b)

for num, want in examples:
    assert Solution().maxDiff(num) == want
print("2 ví dụ: OK")

2 ví dụ: OK


## 10. Vì sao thuật toán đúng

Mọi kết quả hợp lệ có **cùng số chữ số** `d` với `num` (không có số 0 đứng đầu, và thao tác thay chữ số không đổi độ dài). Với số cùng độ dài, so sánh giá trị = so sánh chuỗi từ trái sang phải.

**Max.** Gọi `i` = vị trí đầu tiên `s[i] ≠ '9'` (nếu không có, `num` đã lớn nhất). Mọi kết quả `t` có `t[0..i−1]`: các vị trí này trong `s` là `9`, có thể bị đổi thành nhỏ hơn nhưng không lớn hơn ⇒ `t[0..i−1] ≤ "9…9"`. Tại `i`, `t[i] ≤ 9`. Phương án `s[i] → 9` đạt `"9…9"` ở `0..i` và các vị trí sau chỉ **tăng hoặc giữ nguyên** (chữ số bằng `s[i]` thành 9). Mọi `t` khác hoặc nhỏ hơn ở một vị trí `≤ i`, hoặc bằng ở `0..i` — điều này buộc `x = s[i]`, `y = 9` (hoặc `s[i] = 9`, mâu thuẫn) ⇒ chính là phương án của ta. Vậy tối đa.

**Min.** Nếu `s[0] ≠ '1'`: `t[0] ≥ 1` với mọi `t` hợp lệ; đổi `s[0] → 1` đạt `t[0] = 1`, và để `t[0] = 1` thì buộc `x = s[0], y = 1` ⇒ phương án duy nhất đạt chữ số đầu nhỏ nhất ⇒ tối thiểu. Nếu `s[0] = '1'`: không được đổi `x = 1` (đổi thành 0 ⇒ cấm; thành `y > 1` ⇒ lớn hơn). Gọi `j` = vị trí đầu tiên mà `s[j] ∉ {0, 1}`. Các vị trí trước `j` là 0 hoặc 1 — không giảm được (`0` đã nhỏ nhất, `1` không được đổi). Đổi `s[j] → 0` làm `t[j] = 0` nhỏ nhất, phần sau chỉ giảm hoặc giữ ⇒ tối thiểu. Không có `j` ⇒ không giảm được gì ⇒ `b = num`.

`a`, `b` độc lập ⇒ `max(a − b) = max a − min b`.

## 11. Phân tích Time / Space

Gọi `d` = số chữ số của `num` (`d ≤ 9`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute 100 cặp | `O(100 · d)` | `O(d)` | đủ nhanh với ràng buộc |
| 2. Tham lam chuỗi | `O(d)` | `O(d)` | 2 lần `replace` |
| 3. Tham lam số học | `O(d)` | `O(d)` | không dùng chuỗi |
| 4. Brute có định hướng | `O(30 · d)` | `O(d)` | ít lý luận, khó sai |

Vì `d ≤ 9`, mọi cách đều là "hằng số" trong thực tế; khác biệt chỉ đáng kể ở biến thể chuỗi rất dài.

In [16]:
rng = random.Random(1432)
nums = [rng.randint(1, 10**8) for _ in range(20000)]
for name, f in [("brute", max_diff_brute), ("greedy", max_diff_greedy), ("math", max_diff_math), ("targeted", max_diff_targeted)]:
    t = time.perf_counter(); acc = sum(f(n) for n in nums)
    print(f"{name:9s}: 20000 số ngẫu nhiên ≤ 10^8: {time.perf_counter() - t:.3f}s (checksum {acc})")

brute    : 20000 số ngẫu nhiên ≤ 10^8: 0.351s (checksum 1488735730903)
greedy   : 20000 số ngẫu nhiên ≤ 10^8: 0.010s (checksum 1488735730903)
math     : 20000 số ngẫu nhiên ≤ 10^8: 0.031s (checksum 1488735730903)
targeted : 20000 số ngẫu nhiên ≤ 10^8: 0.071s (checksum 1488735730903)


## 12. Bộ test

1. Ví dụ đề.
2. Edge: `1` (đáp án `8`: `9 − 1`), `10^8 = 100000000`, toàn `9`, toàn `1` (`111 → 999 − 111`), số bắt đầu bằng `1` theo sau là `0` và `1` (`101010`), số có `1` ở giữa nhưng chữ số đầu khác 1.
3. Đối chiếu **toàn bộ** `num ∈ [1, 20000]` và ngẫu nhiên tới `10^8` giữa mọi cách với brute force.

In [17]:
impls = {
    "brute": max_diff_brute, "greedy": max_diff_greedy, "math": max_diff_math, "targeted": max_diff_targeted,
    "Solution": lambda n: Solution().maxDiff(n),
    "SolGreedy": lambda n: SolutionGreedy().maxDiff(n),
    "SolMath": lambda n: SolutionMath().maxDiff(n),
}
edge = [(1, 8), (10**8, 800000000), (999, 888), (111, 888), (101010, 808080), (9288, 8700),
        (123456, 820000), (10000, 80000), (1101057, 8808050), (5, 8), (19, 89)]
for name, f in impls.items():
    for num, want in examples + edge:
        assert f(num) == want, (name, num, f(num))
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [18]:
for num in range(1, 20001):
    ref = max_diff_brute(num)
    for name, f in impls.items():
        assert f(num) == ref, (name, num)
rng = random.Random(8)
for _ in range(3000):
    num = rng.randint(1, 10**8)
    ref = max_diff_brute(num)
    assert all(f(num) == ref for f in impls.values()), num
print("đối chiếu 1..20000 + 3000 số ngẫu nhiên: OK")

đối chiếu 1..20000 + 3000 số ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đổi chữ số đầu thành `0` để có số nhỏ nhất

In [19]:
def wrong_min_zero(num):
    s = str(num)
    a = max_value(num)
    b = int(s.replace(s[0], "0"))               # SAI: tạo số 0 đứng đầu (có thể ra 0)
    return a - b

for num in [555, 9288, 9]:
    print(num, "| sai:", wrong_min_zero(num), "| đúng:", Solution().maxDiff(num))

555 | sai: 999 | đúng: 888
9288 | sai: 9700 | đúng: 8700
9 | sai: 9 | đúng: 8


**Vì sao sai:** `555 → "000"` = 0 (cấm bằng 0), `9288 → "0288"` có số 0 đứng đầu (cấm). `int()` lặng lẽ bỏ số 0 nên không báo lỗi, ra hiệu quá lớn.

**Sửa:** chữ số đầu chỉ được giảm xuống `1`; các chữ số khác mới được về `0`.

### Lỗi 2 — Khi chữ số đầu là `1`, chọn chữ số khác `0` đầu tiên (quên loại `1`)

In [20]:
def wrong_min_skip_only_zero(num):
    s = str(num)
    if s[0] != "1":
        b = s.replace(s[0], "1")
    else:
        b = s
        for c in s[1:]:
            if c != "0":                          # SAI: phải loại cả '1'
                b = s.replace(c, "0"); break
    return max_value(num) - int(b), b

for num in [1101057, 111, 1215]:
    d, b = wrong_min_skip_only_zero(num)
    print(num, f"| sai: {d} (b = '{b}')", "| đúng:", Solution().maxDiff(num))

1101057 | sai: 9909000 (b = '0000057') | đúng: 8808050
111 | sai: 999 (b = '000') | đúng: 888
1215 | sai: 8280 (b = '1015') | đúng: 8280


**Vì sao sai:** với `1101057`, chữ số khác 0 đầu tiên sau vị trí 0 là `1` ⇒ đổi **mọi** `1` thành `0`, kể cả chữ số đầu ⇒ `"0000057"` có số 0 đứng đầu.

**Sửa:** bỏ qua cả `'0'` và `'1'`: `if c not in "01"`.

### Lỗi 3 — Max: luôn đổi chữ số **đầu** thành 9

In [21]:
def wrong_max_first(num):
    s = str(num)
    a = int(s.replace(s[0], "9"))                 # SAI nếu s[0] đã là 9
    return a - min_value(num)

for num in [9288, 99, 919]:
    print(num, "| sai:", wrong_max_first(num), "| đúng:", Solution().maxDiff(num))

9288 | sai: 8000 | đúng: 8700
99 | sai: 88 | đúng: 88
919 | sai: 808 | đúng: 888


**Vì sao sai:** nếu chữ số đầu đã là 9, đổi `9 → 9` không làm gì. Phải tìm chữ số **đầu tiên khác 9** (`9288` ⇒ đổi `2` ⇒ `9988`).

**Sửa:** `for c in s: if c != '9': ...`.

### Lỗi 4 — Chỉ thay **lần xuất hiện đầu tiên** (`replace(x, y, 1)`)

In [22]:
def wrong_replace_once(num):
    s = str(num)
    a = s
    for c in s:
        if c != "9":
            a = s.replace(c, "9", 1); break        # SAI: đề yêu cầu thay MỌI lần xuất hiện
    b = s
    if s[0] != "1":
        b = s.replace(s[0], "1", 1)
    else:
        for c in s[1:]:
            if c not in "01":
                b = s.replace(c, "0", 1); break
    return int(a) - int(b)

for num in [555, 1101057]:
    print(num, "| sai:", wrong_replace_once(num), "| đúng:", Solution().maxDiff(num))

555 | sai: 800 | đúng: 888
1101057 | sai: 8000050 | đúng: 8808050


**Vì sao sai:** đề nói "replace **all** the occurrences of x". Với `555`, thay một lần chỉ ra `955` và `155`, hiệu `800` thay vì `888`. (Thú vị: thay một lần lại có thể cho hiệu lớn hơn ở số khác — nhưng đó là bài khác!)

**Sửa:** `s.replace(x, y)` không có tham số `count`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai lần **độc lập**, tối đa hoá hiệu" ⇒ **max riêng, min riêng**.
- Không gian lựa chọn nhỏ (100 cặp chữ số) ⇒ **brute force là lời giải hợp lệ**; viết nó trước, rồi dùng để kiểm tra tham lam.
- Bài "đổi chữ số để số lớn/nhỏ nhất" ⇒ tham lam **từ trái sang phải** (trọng số lớn nhất).
- Ràng buộc **không có số 0 đứng đầu** ⇒ chữ số đầu xử lý riêng (min là 1, không phải 0).
- Test nhanh: `1`, `9`, `111`, `999`, `10^8`, `1101057`, `9288`.

## 15. Biến thể của bài

1. **LeetCode 2566 Maximum Difference by Remapping a Digit**: **cho phép** số 0 đứng đầu ⇒ min chỉ cần đổi chữ số đầu thành 0.
2. **LeetCode 1323 Maximum 69 Number**: chỉ đổi **một** chữ số `6 → 9` ⇒ đổi chữ số 6 đầu tiên.
3. **LeetCode 670 Maximum Swap**: hoán đổi hai chữ số một lần ⇒ tham lam với vị trí cuối của mỗi chữ số.
4. **Thay chỉ một lần xuất hiện** (thay vì mọi lần) ⇒ max: chữ số đầu tiên khác 9 thành 9; min: tương tự nhưng xét từng vị trí.
5. **Số rất dài dạng chuỗi** (10^5 chữ số) ⇒ tham lam `O(d)` vẫn chạy tốt, brute 100 cặp cũng được nhưng tốn bộ nhớ hơn.

Code biến thể 2 (LeetCode 1323):

In [23]:
def maximum69(num):
    return int(str(num).replace("6", "9", 1))

print(maximum69(9669), maximum69(9996), maximum69(9999))

9969 9999 9999


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute 100 cặp | Thử mọi `(x, y)`, lọc 0 đứng đầu, max − min | `O(100·d)` | `O(d)` | Nộp được ngay; kiểm chứng |
| 2. Tham lam chuỗi | Max: đầu tiên ≠ 9 → 9; Min: đầu → 1 hoặc đầu tiên ∉{0,1} → 0 | `O(d)` | `O(d)` | Lời giải chuẩn, ngắn |
| 3. Tham lam số học | Như 2 nhưng dùng `divmod` | `O(d)` | `O(d)` | Khi cấm dùng chuỗi |
| 4. Brute có định hướng | `x` có mặt, `y ∈ {0,1,9}` | `O(30·d)` | `O(d)` | Ít lý luận, khó sai |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().maxDiff(123456)` bằng bao nhiêu?

### Bài 2 (tính tay)
Tính tay `a`, `b` và đáp án cho `num = 10000`, `num = 9288`, `num = 1110`.

### Bài 3 (điền TODO)
Hoàn thiện `max_value_todo(num)` (tìm `a` lớn nhất) bằng cách duyệt chuỗi.

### Bài 4 (tự viết lại)
Viết `min_value_math(num)` chỉ dùng số học (không `str`), theo quy tắc min.

### Bài 5 (mini challenge)
Viết `min_max_difference(num)` cho **LeetCode 2566**: một lần đổi chữ số (mọi lần xuất hiện), **cho phép** số 0 đứng đầu; trả `max − min`. Kiểm chứng bằng brute force.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_value_todo(num):
    s = str(num)
    # TODO: tìm chữ số đầu tiên khác '9', thay mọi lần xuất hiện bằng '9'
    # TODO: nếu không có, trả num
    pass

# Bài 4 — TODO
def min_value_math(num):
    # TODO: tách chữ số bằng divmod; xử lý chữ số đầu khác 1 / bằng 1
    pass

# Bài 5 — TODO
def min_max_difference(num):
    # TODO: max như cũ; min: đổi mọi chữ số bằng s[0] thành '0' (được phép 0 đứng đầu)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Max: chữ số đầu `1 ≠ 9` ⇒ `923456`. (2) Min: chữ số đầu là `1`; chữ số đầu tiên ∉{0,1} là `2` ⇒ `103456`. (3) `923456 − 103456 = 820000`.

**Bài 2.** (1) `10000`: `a = 90000`, `b = 10000` (không có chữ số giảm được) ⇒ `80000`. (2) `9288`: `a = 9988`, `b = 1288` ⇒ `8700`. (3) `1110`: `a = 9990`; min: sau chữ số đầu chỉ có `1` và `0` ⇒ `b = 1110` ⇒ `8880`.

**Bài 3.** (1) `for c in s:`. (2) `if c != '9': return int(s.replace(c, '9'))`. (3) Sau vòng lặp `return num`.

**Bài 4.** (1) `ds = digits_of(num)`. (2) `ds[0] != 1` ⇒ `remap(ds, ds[0], 1)`. (3) Ngược lại tìm `d > 1` đầu tiên trong `ds[1:]` ⇒ `remap(ds, d, 0)`; không có ⇒ `num`.

**Bài 5.** (1) Max giống `max_value`. (2) Min: `int(s.replace(s[0], '0'))` — chữ số đầu luôn là lựa chọn tốt nhất để về 0. (3) Brute: thử 100 cặp, **không** lọc 0 đứng đầu.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().maxDiff(123456) == 820000
# Bài 2
assert (max_value(10000), min_value(10000), Solution().maxDiff(10000)) == (90000, 10000, 80000)
assert (max_value(9288), min_value(9288), Solution().maxDiff(9288)) == (9988, 1288, 8700)
assert (max_value(1110), min_value(1110), Solution().maxDiff(1110)) == (9990, 1110, 8880)

# Bài 3
def max_value_todo_sol(num):
    s = str(num)
    for c in s:
        if c != "9":
            return int(s.replace(c, "9"))
    return num

# Bài 4
def min_value_math_sol(num):
    ds = digits_of(num)
    if ds[0] != 1:
        return remap(ds, ds[0], 1)
    for d in ds[1:]:
        if d > 1:
            return remap(ds, d, 0)
    return num

# Bài 5
def min_max_difference_sol(num):
    s = str(num)
    return max_value(num) - int(s.replace(s[0], "0"))

def brute_2566(num):
    s = str(num)
    vals = [int(s.replace(x, y)) for x in "0123456789" for y in "0123456789"]
    return max(vals) - min(vals)

assert min_max_difference_sol(11891) == 99009
assert min_max_difference_sol(90) == 99

for num in list(range(1, 5001)) + [random.Random(i).randint(1, 10**8) for i in range(2000)]:
    assert max_value_todo_sol(num) == max(all_results(num))
    assert min_value_math_sol(num) == min(all_results(num))
    assert min_max_difference_sol(num) == brute_2566(num)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam đổi chữ số:**

```python
s = str(num)
# max: chữ số đầu tiên khác 9 -> 9
a = next((s.replace(c, "9") for c in s if c != "9"), s)
# min: chữ số đầu -> 1; nếu đã là 1: chữ số đầu tiên không thuộc {0,1} -> 0
if s[0] != "1":
    b = s.replace(s[0], "1")
else:
    b = next((s.replace(c, "0") for c in s[1:] if c not in "01"), s)
return int(a) - int(b)
```

**Template brute force 100 cặp:**

```python
vals = [int(t) for x in D for y in D if (t := s.replace(x, y))[0] != "0"]
return max(vals) - min(vals)
```

**Checklist:**
- [ ] `a`, `b` độc lập ⇒ max riêng, min riêng.
- [ ] Không tạo số 0 đứng đầu (chữ số đầu min là 1).
- [ ] Khi chữ số đầu là 1, bỏ qua cả `0` **và** `1`.
- [ ] Max: chữ số **đầu tiên khác 9**, không phải chữ số đầu.
- [ ] `replace` thay **mọi** lần xuất hiện.
- [ ] Đã test `1`, `9`, `111`, `999`, `10^8`, `1101057`, `9288`.